# Python Demo
**Before starting: select Python kernel**
- e.g. via command palette: `> Python: Select interpreter` -> choose the environment you created before

**Goals**:
- Use the Card (1993) IV-example (see Exercise 8 of the first part of the lecture), to show that familiar Stata commands (`regress`, `ivregress 2sls`) are, at least in their simplest form, thin wrappers around a few lines of linear algebra.
- Below, we will first use available packages (`statsmodels`, `linearmodels`), and then derive the estimators ourselves and validate them against the packages.

**Structure**
1. Packages for OLS and IV/2SLS (similar to what you've seen in Stata)
2. OLS in Matrix Form
3. Three ways of doing 2SLS:
    - Two explicit OLS stages
    - One-line projection formula
    - GMM-root optimization


**Two important packages**
- `numpy`: all things linear algebra
- `pandas`: data frame layer (data manipulation similar to what you know from Stata)

**Background:** Card (1993) - "Using geographic variation in college proximity to estimate the returns to schooling"
- IV estimator to estimate the causal returns to schooling on hourly log wages
- NLSY dataset: 3k young males
- **Problem**: endogeneity of `educ`. Ability (unobserved) raises both schooling and wages, so $\mathrm{Cov}(educ, u)\neq 0$ and OLS is biased 
- **Card's instrument:** `nearc4` = grew up near a 4-year college. The argument: proximity lowers the cost of schooling (relevance) but does not directly affect wages given the controls (exclusion).
- Details see Exercise 8 (Part 1 of the lecture)

In [ ]:
# Load packages
import numpy as np              # numerical/linear-algebra library; e.g. provides the ndarray and `@` (matrix multiplication) operator
import pandas as pd             # data-frame library; the in-memory equivalent of a Stata dataset

# Load packages for regression analysis
import statsmodels.api as sm
from linearmodels.iv.model import IV2SLS

# For details, review the documenation:
#   - numpy: https://numpy.org/doc/stable/ 
#   - pandas: https://pandas.pydata.org/docs/user_guide/index.html
#   - statsmodels: https://www.statsmodels.org/stable/index.html
#  - linearmodels: https://pypi.org/project/linearmodels/

## 1. Redoing Card's IV Example - With Packages
- Use the estimators provided in the packages: OLS and IV/2SLS
    > For details about syntax, expected inputs and executed procedures, review the documentation of the packages.
- Set up the validation targets for our own estimators: `ols_reg`, `iv_reg`


### 1.a Load and review data:

In [ ]:
# First load data:
# The data is in Stata format, so we use the `read_stata` function from pandas.

card = pd.read_stata("card.dta")

In [ ]:
# Review the data:
print(card.shape)  # number of rows and columns
card.head()

In [ ]:
# Get a concise summary of the data, including variable names, non-missing values, and descriptives:
card.describe()

In [ ]:
# Get descriptive for subset of variables:

# directly:
# card[["lwage", "educ", "exper"]].describe()

# first define a 'list' of variables:
vars = [
    "lwage", # dependent var
    "educ",  # years of education
    "exper"
    ]

card[vars].describe()

In [ ]:
# Histogram of years of education:
card["educ"].hist()

In [ ]:
# Kernel Density of log-wages:
card["lwage"].plot.kde()

# review `matplotlib` documentation for more plotting options:
# https://matplotlib.org/stable/contents.html

### 1.b) Run OLS
- Use `form_formula` from the `statsmodels` package: R/patsy formula syntax (`y ~ x1 + x2`)
- `.fit()` runs the regression; `.summary()` to plot regression table output

**Result:**
- naive OLS return to schooling (`educ` coefficient): ~7.2% per year of schooling

**Note**
- `from_formula`adds the intercept automatically

In [ ]:
# Stata equivalent:
#   regress lwage educ exper expersq black south married smsa smsa66 reg662-reg669

# Notes on OLS from statsmodel package:
# - `.from_formula` is a class method (called from OLS class sm.OLS)
# - parses the provided regression 'formula', builds design matrix X and adds a intercept automatically
# - `data=` is the DataFrame to read from - here the pandas dataframe we called `card` containing our dataset
# - Methods can be chained (see below): `.from_formula(...)' returns unfitted model, and `.fit()` returns the fitted results in one line

ols_reg = sm.OLS.from_formula("lwage ~ educ + exper + expersq + black + south + married + smsa + smsa66 + reg662 + reg663 + reg664 + reg665 + reg666 + reg667 + reg668 + reg669", 
              data = card).fit()

# .summary() returns the familiar regression table
ols_reg.summary()

#### Same regression, but we build the design matrix X ourselves
- Instead of using the regression formula string, we keep the set of regressors in a Python list `x_var` and construct the 'X' matrix programmatically.
- We can use `sm.add_constant` from the statsmodels package to add a column of ones (the intercept).
- `.dropna()` for list-wise deletion pattern of missing regressors , then `card.loc[X.index, "lwage"]` to line up the dependent variable exactly on the surviving rows - important to make sure `X` and `y` have the same set of rows.

**Output should be identical to results generated above!**

In [ ]:
# Define a Python list (using square brackets) of regressor names as strings (independent variables):
x_vars = [
    "educ",
    "exper",
    "expersq",
    "black",
    "south",
    "married",
    "smsa",
    "smsa66",
    "reg662",
    "reg663",
    "reg664",
    "reg665",
    "reg666",
    "reg667",
    "reg668",
    "reg669"
]


# Generate X container:
# - Indexing a DataFrame with a LIST of column names -> card[x_vars] returns a sub-DataFrame
#   with just those columns (note the double brackets: card[[...]] semantics)
# - sm.add_constant prepend column of ones
X = sm.add_constant(card[x_vars])

# Rename for consistent labeling
X = X.rename(columns={"const": "intercept"})

# Ensure there are no NaNs:
# - run list-wise deletion of rows with missing values
X = X.dropna()

# Pass data matrices directly to estimator (instead of formula-string above):
# - call OLS estimator from statsmodels package: 'endog'/'exog' in statsmodels refers to LHS/RHS vars
#     > Package naming convention =/= econometric endogeneity of e.g. education
# - card.loc[X.index, "lwage"] to use label-based indexing: .loc[row_labels, column] selects 
#   the lwage values on exactly the remaining rows in X from our dataframe `card``
ols_reg = sm.OLS(
    endog=card.loc[X.index, "lwage"],
    exog=X
)
# Run the defined regression and plot output
ols_reg = ols_reg.fit()
ols_reg.summary()


In [ ]:
X

### 1.c) IV/2SLS the package way
- Package `linearmodels` offers IV support/estimators
- Formula syntax similar to above: **brackets for the endogenous-instrument relationship**: `[educ ~ nearc4]` (read: educ is endogenous and instrumented by nearc4)
- Everything outside the brackets is treated as exogenous control entering **both** stages.

**Notes on `linearmodels`:**
- drops missing rows for us - keep in mind for computations further below.
- defaults to **robust** heteroskedasticity-consistent SEs. Our estimator code below will use only classical homoskedastic SEs (which would result in same coefficient estimates, different SEs)
- **degrees-of-freedom convention:** divides by $n$, unless you pass `debiased=True`(divides by $n-k$, matching the Stata command)
- > Thus to ensure comparability to our estimator code below, we pass `cov_type="unadjusted", debiased=True`.


**IV estimation:**
- Endogeneity: $\mathrm{Cov}(educ,u)\neq 0$, so OLS is inconsistent for the causal effect.
- A valid instrument $Z$ satisfies **relevance** $\mathrm{Cov}(Z,educ)\neq0$ and **exogeneity /
  exclusion** $\mathrm{Cov}(Z,u)=0$. IV "uses only the variation in `educ` that is driven by
  `nearc4`" (the part plausibly unrelated to ability).
- Result: `educ` coefficient moves compared to naive OLS result

In [ ]:
# Stata equivalent (robust):
#   ivregress 2sls lwage exper expersq black south married smsa smsa66 \
#       reg662-reg669 (educ = nearc4), vce(robust)

# First create list of x_vars without educ:
x_vars_no_educ = [
    "exper",
    "expersq",
    "black",
    "south",
    "married",
    "smsa",
    "smsa66",
    "reg662",
    "reg663",
    "reg664",
    "reg665",
    "reg666",
    "reg667",
    "reg668",
    "reg669"
]

# Generate the formula for the IV regression:
# - This time, we build the formula string programmatically rather than typing it out:

#   1. Start by adding dependent var and intercept to string: "lwage ~ 1 +"
#   2. " + ".join(list): combines list items together with " + " between them (-> exogen x-vars)
#   (str.join is standard Python syntax to concatenating a list of strings)
#   3. Finally, add the instrument with bracket syntax [endog ~ instruments]
#   -> Each of these string components is joined together using the + in between

# Note that we use the formula syntax from linearmodels
iv_formula = "lwage ~ 1 + " + " + ".join(x_vars_no_educ) + " + [educ ~ nearc4]"

# review results - string of our formula:
iv_formula



In [ ]:
# Run IV regression
iv_reg = IV2SLS.from_formula(iv_formula, card).fit()
iv_reg.summary      # no parantheses .(): .summary is a property, not a method call (linearmodels package)

#### First stage and weak-instrument check

Before trusting the IV estimate we check that the instrument `nearc4` is relevant, i.e. that it actually predicts the endogenous regressor `educ` in the first stage. The standard diagnostic is the first-stage F-statistic on the excluded instrument(s); a common rule of thumb is $F > 10$.


In [ ]:
# First-stage regression: educ on the instrument and all exogenous regressors
first_stage = sm.OLS.from_formula(
    "educ ~ nearc4 + " + " + ".join(x_vars_no_educ),
    data=card,
).fit()

# F-statistic for the excluded instrument (nearc4)
# .f_test takes a restriction string and tests it (here H0: coefficient on nearc4 = 0).
f_test = first_stage.f_test("nearc4 = 0")
print(f"First-stage coefficient on nearc4: {first_stage.params['nearc4']:.4f}")
print(f"First-stage F-statistic (nearc4):  {float(f_test.fvalue):.2f}")

# Stata equivalent:
#   regress educ nearc4 exper expersq black south married smsa smsa66 reg662-reg669
#   test nearc4


## 2. OLS in Matrix Form - manually

**Some (incomplete) notes:**

We start from model

$$y = X \beta + \varepsilon$$

where:
- $\bm{X}$ is a $n\times k$ matrix of regressors (one column is a vector of ones for the constant)
- $\bm{y}$ is a $n \times 1$ vector of observations of the dependent variable.
- $\bm{\varepsilon}$ is a $n \times 1$ vector of unobserved components
- $\bm{\beta}$ is a $k \times 1$ vector of unknown parameters

Recall the **OLS estimator formula**:

$$\hat{\beta} = (X'X)^{-1}X'y$$


**Variance of the OLS estimator**

Under homoskedasticity/no-autocorrelation $E[\varepsilon\varepsilon' \mid X] = \sigma^2 I$, we have

$$Var(\hat{\beta} \mid X) = \sigma^2 (X'X)^{-1}$$

which we can estimate using the estimated disturbance variance

$$\hat{\sigma}^2 = \frac{1}{n-k} e'e$$

$$\hat{Var}(\hat{\beta}) = \hat{\sigma}^2 (X'X)^{-1}$$

where $\bm{e} = \bm{y} - \bm{X}\bm{\hat{\beta}}$.

Note: you can find helpful derivation notes, e.g. [here](https://web.stanford.edu/~mrosenfe/soc_meth_proj3/matrix_OLS_NYU_notes.pdf).


**Below we apply these formulas:**
- Prepare matrices, e.g. $(X'X)^{-1}$
- compute residuals

In [ ]:
# First get the data by 'method-chaining' again
# - get the x_vars from the dataframe using the list we defined
# - add a constant using the statsmodel package
# - drop rows with missing
x = sm.add_constant(card[x_vars]).dropna()

# Get index of x:
# - .index is the row-label object of the (cleaned) dataframe -> again to align y and x/z
x_index = x.index

# Transform x to a matrix
# - .values returns the underlying numpy ndarray (drops column names/index from the dataframe). 
#    -> or use .to_numpy()
# - From here, we do pure linear algebra on a plain matrix, rather than a labelled dataframe
print(f"before: {type(x).__name__}, shape={x.shape}, cols={list(x.columns)}") # inspect before conversion
#x = x.values
x = x.to_numpy()
print(f"after: {type(x).__name__}, shape={x.shape}, dtype={x.dtype}")


# Get the dependent variable -> select lwage to match surviving rows
y = card.loc[x_index, "lwage"]
print(y.shape, type(y))

In [ ]:
# sometimes it is useful to include programmatic sanity checks in your code:

# simple version that spits out message if assertion is false (assert cond, msg)
assert card.loc[x_index, "lwage"].shape[0] == x.shape[0], "row count mismatch"

# alternative 
if card.loc[x_index, "lwage"].shape[0] == x.shape[0]:
    print(f"Row counts match: {x.shape[0]} rows")
else:
    # stops exection and specific error type
    raise AssertionError("row count mismatch")

#### $\hat\beta=(X'X)^{-1}X'y$ in three lines
- `x.T @ x` is $X'X$ (a $k\times k$ matrix); `np.linalg.inv` inverts it; one more multiply by
  $X'y$ gives $\hat\beta$.

**Printed output**:
- first entry is the intercept, then the coefficients in `x_vars` order. 
- the second entry (`educ`) is ~0.0722 — the OLS return again

In [ ]:
# Calculate X'X
# - .T to matrix transpose; @ is the matrix-multiplication operator from numpy 
# (Note `*` would be elementwise multiplication)
xpx = x.T @ x

# Use numpy for the inverse
# - np.linalg is numpy's linear-algebra submodule
# - .inv computes a matrix inverse
xpx_inv = np.linalg.inv(xpx)

# Calculate the OLS estimator
# - parantheses to group X'y  (@ is left-associative, so this is just for clarity)
beta = xpx_inv @ (x.T @ y)

# bare variable name on the last line of a cell auto-displays its value(s) in a Jupyter notebook
beta

In [ ]:
# Print educ coefficient (index 1)
print(f"OLS coefficient on education: {beta[1]}")

#### Check if we recover the OLS estimates of the packages:
- `np.allclose(a,b)` returns `True` if two arrays agree to floating-point tolerance.
- Preferable to using exact comparision `==`, since results from different procedures might produce tiny rounding differences.

In [ ]:
# Check that those are the same as the OLS coefficients
# np.allclose(a, b) -> True if a and b agree within a small numerical tolerance.
np.allclose(ols_reg.params, beta)

#### Packaging OLS into a re-useable function
- Define re-usable function for OLS estimators. We will call this function in the 2SLS estimator below.

In [ ]:
# `def name(args):` defines a function; the indented block is its body;
# `return` hands back the result. This one returns TWO objects as a tuple.

def ols_formula(y, x):
    """
    Define a function for the OLS estimator.

    Args:
        y: dependent var, a 1d array/series (n x 1)
        x: independent vars, a 2d array (n x k) (num_obs, num_xvars)
        
    Returns:
        coeffs: estimated OLS coefficients
        std_errors: estimated standard errors (assuming homosked.)
    """

    # (X'X)^-1
    inverse_covars = np.linalg.inv(x.T @ x)

    # OLS estimator formula
    coeffs = inverse_covars @ (x.T @ y)


    # Add estimation of standard errors:
    # compute linear prediction Xb
    projection = x @ coeffs

    # compute residuals (nx1)
    residuals = y - projection

    # sum of squared residuals: vector-dot product (1d @ 1d -> scalar) = sum of squares
    squared_sum_residuals = residuals @ residuals

    # get degrees of freedom (n-k)
    # - .shape is a (rows, cols) tuple: .shape[0] = n observations (rows of X); .shape[1]= k regressors (cols of X)
    degrees_of_freedom = x.shape[0] - x.shape[1]

    # estimated covariance matrix (e'e)/(n-k) * (X'X)^-1 -> (kxk)
    covariance_est = (squared_sum_residuals / degrees_of_freedom) * inverse_covars
    

    # extract diagonal and take square root to get estimated standard errors
    std_errors = np.sqrt(np.diag(covariance_est))

    # define function returns
    return coeffs, std_errors

In [ ]:
# Use formula and compare results
coeffs, std_errors = ols_formula(y, x)
np.allclose(ols_reg.params, coeffs)

## 3. Re-doing Card IV-example - manually

### IV manually

Let $ y \in \mathbb{R}^n $ be the outcome variable, and let the regressor matrix be partitioned as
$$
X = \begin{bmatrix} X_o & x_e \end{bmatrix},
$$
where:

- $ X_o \in \mathbb{R}^{n \times k} $: matrix of exogenous regressors,
- $ x_e \in \mathbb{R}^{n \times 1} $: endogenous regressor,
- $ Z \in \mathbb{R}^{n \times \ell} $: matrix of instruments, including excluded instruments and the exogenous variables in $ X_o $.

---

#### Two-Stage Least Squares (2SLS) Estimator

**Stage 1 (First Stage):**  
Regress the endogenous regressor $ x_e $ on the instruments $ Z $:
$$
\hat{x}_e = P_Z x_e, \quad \text{where } P_Z = Z (Z^\top Z)^{-1} Z^\top
$$
is the projection matrix onto the column space of $ Z $.

**Stage 2 (Second Stage):**  
Regress $ y $ on the fitted regressor matrix:
$$
\hat{X} = \begin{bmatrix} X_o & \hat{x}_e \end{bmatrix}.
$$
Then the 2SLS estimator is given by:
$$
\hat{\beta}_{2SLS} = \left( \hat{X}^\top \hat{X} \right)^{-1} \hat{X}^\top y.
$$

---


**Background — the correct 2SLS variance**
$$\widehat{\mathrm{Var}}(\hat\beta_{2SLS})=\hat\sigma^2\,(X'P_Z X)^{-1},\qquad
\hat\sigma^2=\frac{(y-X\hat\beta)'(y-X\hat\beta)}{n-k},$$
with $X$ the **original** regressor matrix. The code's `bread` $=(X'P_Z X)^{-1}$ and the
residuals built from `x_orig` implement exactly this.


In [ ]:
def two_sls_with_OLS(y, x_exog, x_to_instrument, z):
    """
    Two stage least squares estimator (estimated via two explicit OLS steps).

    Args:
        y: dependent variable, 1d array/series
        x_exog: exogenous regressors (everything EXCEPT the endogenous regressor)
        x_to_instrument: the endogenous regressor (instrumented in the first stage)
        z: instruments, 2d array (exogenous regressors + excluded instrument(s))

    Returns:
        coeffs: estimated coefficients (second stage)
        std_errors: classical (homosked.) standard errors    


    NOTE: The second-stage SEs are NOT the naive OLS SEs from regressing y on the
    fitted regressors. The variance must be computed using residuals based on the
    ORIGINAL endogenous regressor (x_to_instrument), not its first-stage fitted
    values. Using the fitted values would understate the residual variance and give
    wrong standard errors. 
    """

    # First stage: regress the endogenous regressor on the instruments z
    # - re-use the function ols_formula we defined above.
    # - ", _" discards the output of the estimated standard errors
    first_stage_ols_coeff, _ = ols_formula(x_to_instrument, z)

    # Get the fitted values using first-stage estimates
    x_hat = z @ first_stage_ols_coeff
    


    # Second stage: regress y on [exogenous regressors, x_hat]
    # - first combine the set of regressors:
    #    - np.hstack: stacks arrays side-by-side (horizontally) into one matrix
    #    - recall x_hat is 1d (shape(n,)); reshape(-1,1) makes it a column (n,1), then we can add it (-1: means infer this dimension from the data) 
    x_total = np.hstack((x_exog, x_hat.reshape(-1, 1)))

    # Run OLS again:
    coeffs, _ = ols_formula(y, x_total)


    # Standard errors: rebuild the regressor matrix with the ORIGINAL endogenous
    # column and use the projection P_z to form (X' P_z X)^{-1}.
    x_orig = np.hstack((x_exog, x_to_instrument.reshape(-1, 1)))
    # compute projection matrix P_z = Z(Z'Z)^-1 Z'
    pz = z @ np.linalg.inv(z.T @ z) @ z.T
    # sandwich
    bread = np.linalg.inv(x_orig.T @ pz @ x_orig)

    # Compute estimated standard errors
    residuals = y - x_orig @ coeffs                 # residuals use ORIGINAL educ
    dof = x_orig.shape[0] - x_orig.shape[1]
    sigma2 = (residuals @ residuals) / dof
    cov = sigma2 * bread
    std_errors = np.sqrt(np.diag(cov))
    
    return coeffs, std_errors


#### Run function `two_sls_with_OLS`
- Get the endogenous column of `educ`, and the exogenous block
- Build matrix Z (=exogen. controls + `nearc4`)


In [ ]:
# Read out the instruments Z: (exog. regressors + instrument)
# - card.loc[x_index]: restrict data to the columns we identified before using x_index
# - [x_vars_no_educ + ["nearc4"]: selection of columns - by concatenation of lists: list + list
#     > we get: x_vars_no_educ + ["nearc4"] (controls plus instrument)
# - trailing .values converts to numpy array
z = sm.add_constant(card.loc[x_index][x_vars_no_educ + ["nearc4"]]).values

# Read out the endogenous regressor: educ
x_to_instrument = card.loc[x_index]["educ"].values

# Read out the exogenous regressors (standard controls used before)
# - here we already add a constant again because these are used in the second stage!
x_without_educ = sm.add_constant(card.loc[x_index][x_vars_no_educ]).values


# Calculate the 2SLS coefficients and (classical) standard errors
coeffs_2sls, se_2sls = two_sls_with_OLS(y, x_without_educ, x_to_instrument, z)


# Point estimates match the package exactly
print("Coefficients match package:", np.allclose(iv_reg.params, coeffs_2sls))

# Classical standard errors do not match
print("Try 1: Classical SEs match package:", np.allclose(iv_reg.std_errors, se_2sls))



#### As noted above: Classical SE do not match, because the package computes heterosked. robust errors (and the used degrees of freedom convention)

In [ ]:
# The classical SEs match the package ONLY if the package also uses classical SEs.
# iv_reg above was fit with the default robust covariance, so they will differ.
#
# There is also a degrees-of-freedom convention to reconcile: our manual SEs divide
# the residual variance by (n - k), the small-sample correction that Stata's
# ivregress also uses. linearmodels divides by n unless debiased=True is set.
# We therefore refit with BOTH cov_type="unadjusted" (classical, not robust) and
# debiased=True (the n - k correction) to compare like-for-like:
iv_reg_classical = IV2SLS.from_formula(iv_formula, card).fit(
    cov_type="unadjusted", debiased=True
)
print("Coefficients match package:", np.allclose(iv_reg.params, coeffs_2sls))
print("Classical SEs match package:",
      np.allclose(iv_reg_classical.std_errors, se_2sls))



#### Alternative Matrix Formulation - Collapse two stages in one line

We can also express the 2SLS estimator using the projection matrix $ P_Z $ directly:
$$
\hat{\beta}_{2SLS} = \left( X^\top P_Z X \right)^{-1} X^\top P_Z y,
$$
where $ X = \begin{bmatrix} X_o & x_e \end{bmatrix} $ includes the endogenous regressor before projection. Here, $ P_Z $ again denotes the projection matrix:
$$
P_Z = Z (Z^\top Z)^{-1} Z^\top.
$$

---

In [ ]:
def two_sls_formula(y, x, z):
    """
    Matrix version of the two stage least squares estimator.

    y: dependent variable (n,)
    x: regressors INCLUDING the endogenous regressor (n, k)
    z: instruments = exogenous regressors + excluded instrument(s) (n, l)

    Returns the coefficients and classical (homoskedastic) standard errors.
    """
    pz = z @ np.linalg.inv(z.T @ z) @ z.T            # projection onto col(Z)
    bread = np.linalg.inv(x.T @ pz @ x)
    coeffs = bread @ (x.T @ pz @ y)

    # Classical 2SLS variance: sigma^2 * (X' P_z X)^{-1},
    # with residuals formed from the ORIGINAL X (not the projected X).
    residuals = y - x @ coeffs
    dof = x.shape[0] - x.shape[1]
    sigma2 = (residuals @ residuals) / dof
    cov = sigma2 * bread
    std_errors = np.sqrt(np.diag(cov))
    
    return coeffs, std_errors


In [ ]:
x_iv = sm.add_constant(card.loc[x_index][x_vars_no_educ + ["educ"]]).values
coeff_iv, se_iv = two_sls_formula(y, x_iv, z)

# Compare with the IV regression (point estimates, and classical SEs)
print("Coefficients match package:", np.allclose(iv_reg.params, coeff_iv))
print("Classical SEs match OLS-based 2SLS:", np.allclose(se_iv, se_2sls))


#### GMM Criterion Formulation of 2SLS

The 2SLS estimator can also be derived as a special case of the Generalized Method of Moments (GMM). The **population moment condition** is:
$$
\mathbb{E} \left[ Z^\top (y - X\beta) \right] = 0.
$$
i.e. orthogonality condition on instruments and errors. At the *true* parameter $\beta$, **exogeneity** assumption implies the instruments should be uncorrelated with unobservables/structural error.

The corresponding (vector of) **sample moments** is (replacing the expectation with the sample average):
$$
g_n(\beta) = \frac{1}{n} Z^\top (y - X\beta).
$$
**Goal:** find $\beta$ (vector $k$ parameters) making this expression close to zero. Note that $g_n(\beta)$ has size $(l \times 1)$.

The **GMM criterion function** (which we want to minimize, or find the root) is:
$$
\hat{\beta} = \argmin_{\beta} Q_n (\beta) \qquad , \qquad Q_n(\beta) = g_n(\beta)^\top W g_n(\beta),
$$
where $ W $ is a **symmetric positive definite weighting matrix**. 'Quadratic form' gives a **scalar weighted distance measure**.





Note that $ W $ allows us to account for the covariance structure of the moments (moment conditions can have different variances, be correlated with each other). The weighting matrix lets us scale and combine them correctly.

For **2SLS**, the efficient choice (under homoskedasticity) is
$$
W = (Z^\top Z)^{-1},
$$
which yields $\hat{\beta}_{GMM} = (X^\top P_Z X)^{-1} X^\top P_Z y$, i.e. exactly the 2SLS estimator above.

Note, however, that our example is **just-identified**:
- there is one excluded instrument (`nearc4`) for one endogenous regressor (`educ`), so $\dim(Z) = \dim(X)$
- In this case the sample moment condition $g_n(\beta) = 0$ can be solved exactly, the criterion attains $Q_n = 0$ at the solution -> $k=l$ (params/equations)
- AND the choice of $W$ is irrelevant — any positive definite $W$ gives the same $\hat{\beta}$. (just scaling)
- The weighting matrix only matters under **over-identification** (more excluded instruments than endogenous regressors). The code below therefore solves the moment condition directly by root-finding rather than minimizing $Q_n$.

In [ ]:
def gmm_moment_condition(beta, y, x, z):
    """
    GMM moment condition for 2SLS:
        g(beta) = Z' (y - X beta)
    The root of g(beta) gives the 2SLS estimate.
    
    Parameters:
        beta: (k,) array, guess for coefficients
        y: (n,) array, dependent variable
        x: (n, k) array, regressors (may include endogenous)
        z: (n, l) array, instruments = exogenous regressors + excluded instrument(s)

    NOTE: in the just-identified case here: k=l    

    Returns:
        moments: (l,) array, the sample moment vector
    """
    residuals = y - x @ beta
    moments = z.T @ residuals / len(y)
    
    return moments

In [ ]:
from scipy.optimize import root

# Example usage
# root(func, x0=..., args=...) finds beta such that func(beta, *args) == 0.
#   x0       : starting guess. np.zeros(n) makes a length-n vector of zeros;
#              x.shape[1] is the number of coefficients k.
#   args     : a TUPLE of the EXTRA arguments passed to func after `beta`,
#              i.e. func(beta, y, x_iv, z). The order must match the function signature.
res = root(gmm_moment_condition, x0=np.zeros(x.shape[1]), args=(y, x_iv, z))

# Always verify the solver converged before trusting res.x
assert res.success, f"root finder did not converge: {res.message}"
res.success

In [ ]:
# Compare with the IV regression (point estimates only; the just-identified
# moment condition pins down beta but does not itself produce standard errors).
np.allclose(iv_reg.params, res.x)


## Summary of the Three Estimators
\noindent The three estimators implemented in the demo are one object viewed three ways:
\begin{align}
\text{OLS:} \quad & \hat{\beta} = (\bm{X}^\top \bm{X})^{-1} \bm{X}^\top \bm{y}, \\[2pt]
\text{2SLS:} \quad & \hat{\beta} = (\bm{X}^\top \bm{P}_Z \bm{X})^{-1} \bm{X}^\top \bm{P}_Z \bm{y},
                     \qquad
                     \bm{P}_Z = \bm{Z}(\bm{Z}^\top \bm{Z})^{-1} \bm{Z}^\top, \\[2pt]
\text{GMM:} \quad & \hat{\beta} = \arg\min_{\beta}
                     \big[\bm{Z}^\top(\bm{y}-\bm{X}\beta)\big]^\top
                     (\bm{Z}^\top \bm{Z})^{-1}
                     \big[\bm{Z}^\top(\bm{y}-\bm{X}\beta)\big].
\end{align}

- OLS is a special case of 2SLS with $\bm{Z} = \bm{X}$;
- 2SLS is GMM with $\mathbf{W} = (\bm{Z}^\top \bm{Z})^{-1}$; 
- and in the just-identified case all of them reduce to $(\bm{Z}^\top \bm{X})^{-1} \bm{Z}^\top \bm{y}$.

### Interpretation

Comparing the two estimates of the return to schooling:

- **OLS:** the coefficient on `educ` is about **0.072** (≈ 7.2% per year of schooling).
- **IV (2SLS):** the coefficient on `educ` is about **0.120** (≈ 12% per year).

The IV estimate is *larger* than OLS. Two standard explanations:

1. **Measurement error / attenuation:** classical measurement error in `educ` biases OLS toward zero; IV corrects for it, pushing the estimate up.
2. **LATE interpretation:** with a binary instrument (proximity to a college, `nearc4`), 2SLS recovers a Local Average Treatment Effect — the return for *compliers*, individuals induced to acquire more schooling because they grew up near a college. These tend to be individuals from more disadvantaged backgrounds with high marginal returns to education, so the LATE can exceed the OLS average.

The first-stage F-statistic computed above (≈ 12) clears the conventional rule-of-thumb threshold of 10, so `nearc4` is a relevant — though not especially strong — instrument. The relatively modest F is itself worth noting: the wide IV confidence interval reflects this.
